In [22]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

ENV_PATH = './beauty-agent/.env'

# .env 파일에서 환경 변수 로드
load_dotenv(ENV_PATH)

True

In [23]:
# init_chat_model 사용 (권장)
llms = [
    init_chat_model("openai:gpt-5-mini", temperature=0),
    init_chat_model("google_genai:gemini-2.5-flash", temperature=0)
    ]

In [27]:
gpt_response = llms[0].invoke([HumanMessage(content="LangGraph의 핵심 개념을 설명해주세요.")])
print(gpt_response.content)

어떤 맥락의 LangGraph를 말씀하시는지 확인하고 싶습니다. 특정 라이브러리나 논문(예: GitHub 프로젝트 링크)이 있다면 알려주시면 그에 맞춰 정확히 설명드릴게요.  
일반적으로 “LangGraph”라는 이름으로 얘기될 수 있는 개념(즉, 언어 모델(LLM)과 그래프 구조(지식 그래프·관계 그래프)를 결합한 시스템)의 핵심 개념을 정리해 드리면 다음과 같습니다.

핵심 개념 요약
- 그래프 구조(데이터 모델)
  - 노드(nodes), 엣지(edges), 라벨/타입, 속성(properties). 엔터티와 관계를 구조화해서 표현.
  - 온톨로지/스키마: 노드·관계 유형과 제약을 정의해 일관성 유지.

- 그래프 구축(수집·정규화)
  - 정보추출(IE): 텍스트에서 엔터티 추출(NER), 관계 추출, 이벤트 추출.
  - 정규화/엔티티링킹(entity linking): 같은 실체를 하나의 노드로 통합(동일성 해결).
  - 증거·출처 연결: 각 노드/엣지에 소스와 신뢰도 메타데이터 보관.

- 임베딩과 벡터화
  - 노드·문서·문장 임베딩으로 의미적 유사성 측정.
  - 그래프 + 벡터 하이브리드 검색: 구조적 쿼리(경로, 패턴) + 의미 검색(유사도).

- 검색 및 맥락 구성(RAG 스타일)
  - LLM에 줄 맥락(context)을 구성할 때, 서브그래프나 관련 노드 집합을 검색해 제공.
  - 서브그래프 선택 전략: 중요도, 연결도, 신뢰도 기반 필터링.

- 질의 및 탐색
  - 그래프 쿼리 언어(Cypher, SPARQL 등) 또는 API로 패턴 매칭·경로 탐색 수행.
  - 그래프 탐색을 LLM의 추론 과정과 결합(예: "다음으로 무엇을 조회할지"를 LLM이 결정).

- 추론 및 논리
  - 규칙 기반 추론(논리규칙, 트리플 추론)과 통계적 추론(GNN, 임베딩 기반) 병행.
  - LLM을 이용한 체계적 추론 보조(체인 오브 생각, 증거 기반 응답 생성).

- 에이전트·도구 통합
  - LLM 에이전트가 그래프 조회·수정(노드 추가/

In [24]:
gemini_response = llms[1].invoke([HumanMessage(content="LangGraph의 핵심 개념을 설명해주세요.")])
print(gemini_response.content)

LangGraph는 LangChain을 기반으로 구축된 라이브러리로, **LLM 기반 애플리케이션의 복잡한 제어 흐름과 상태 관리를 그래프 형태로 정의하고 실행할 수 있게 해줍니다.** 특히, 여러 단계를 거치며 스스로 판단하고 행동하는 '에이전트(Agent)' 시스템을 구축하는 데 최적화되어 있습니다.

기존 LangChain 체인이나 런너는 주로 선형적인 흐름을 따릅니다. 하지만 실제 복잡한 LLM 애플리케이션, 특히 에이전트는 특정 조건에 따라 다른 경로로 이동하거나, 이전 단계의 결과를 바탕으로 다시 판단하고 행동하는 '순환(loop)' 구조가 필요합니다. LangGraph는 이러한 복잡한 제어 흐름을 명확하고 유연하게 정의할 수 있도록 돕습니다.

---

### LangGraph의 핵심 개념

LangGraph의 작동 방식을 이해하기 위한 주요 개념들은 다음과 같습니다.

1.  **그래프 (Graph):**
    *   LangGraph의 가장 기본적인 단위는 '그래프'입니다. 이 그래프는 '노드(Node)'와 '엣지(Edge)'로 구성됩니다. 마치 흐름도(flowchart)와 같습니다.

2.  **노드 (Node):**
    *   그래프 내에서 특정 작업을 수행하는 단위입니다. 각 노드는 파이썬 함수, LLM 호출, 다른 LangChain Runnable, 또는 심지어 또 다른 에이전트가 될 수 있습니다.
    *   노드는 입력(input)을 받아 처리하고, 결과(output)를 반환합니다.

3.  **엣지 (Edge):**
    *   노드와 노드를 연결하여 데이터와 제어 흐름의 방향을 정의합니다. 엣지는 크게 두 가지 유형이 있습니다:
        *   **직접 엣지 (Direct Edge):** 특정 노드에서 다음 노드로 무조건 이동합니다.
        *   **조건부 엣지 (Conditional Edge):** 소스 노드의 출력에 따라 여러 가능한 다음 노드 중 하나를 선택하여 이동합니다. 에이전트가 '생각하고 판단'

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 1️⃣ State: 카운터를 저장하는 상자
class CounterState(TypedDict):
    count: int

# 2️⃣ Node: 카운터를 증가시키는 함수
def increment(state):
    print(f"현재 카운트: {state['count']}")
    new_count = state["count"] + 1
    print(f"새로운 카운트: {new_count}")
    return {"count": new_count}     # new state 반환

# 3️⃣ Edge: 노드들을 연결하는 그래프
graph = StateGraph(CounterState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# 실행해보기
app = graph.compile()
result = app.invoke({"count": 0})
print(f"최종 결과: {result}")


현재 카운트: 0
새로운 카운트: 1
최종 결과: {'count': 1}


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot_node(state: State) -> dict:
    llm = init_chat_model("google_genai:gemini-3-flash-preview", temperature=0)
    response = llm.invoke(state['messages'])
    return {"messages": [response]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile()

result = graph.invoke({"messages": [{"role": "user", "content": "Hello!"}]})
print(result)

{'messages': [HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='0300e542-59e4-4d41-97ec-df051c43d0c0'), AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvEBCu4BAQw51scfblTAUz7f1DNT4aEEM01VTgoCNHeTyYYzzlh83KrKVLbJ4IVEosT/OCre54lHNsieRFPmyaPjKHqRYNMUdArChLA79gT+oHRvD20sPtELK5YJaDOR4gmssPMBXaalNajcRfKXzOCO3n5fyTrhhlHDojJFbFxtl1Xye7ujoi45Ao4EeE9OmgX2jrqfo2i2P0QpNrbDMokFlCkNbv1xzEkhAYA12m/gvE8M5unX0SThRcNmyFmSd2c83PxZcjAfzLLMpJ1LGE8xreFjE2nxZr8RfXN0MdrmaO3XqrC9Kx9L8JMcDOclX2+B3w=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e38bd-3d1e-7c90-81e0-6119152f7928-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 58, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 49}})]}
